In [3]:
from transformers import AutoTokenizer, AutoModel
import torch
print("Libraries loaded ✅")

Libraries loaded ✅


In [5]:
print("Downloading MuRIL tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")
print("Tokenizer loaded ✅")

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

C:\Users\Kshitij\lekhak\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kshitij\.cache\huggingface\hub\models--google--muril-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Tokenizer loaded ✅


In [6]:
print("Downloading MuRIL model (~900MB, takes a few minutes)...")
model = AutoModel.from_pretrained("google/muril-base-cased")
model.eval()
print("Model loaded ✅")

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded ✅


model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

In [7]:
sentences = [
    "मी शाळेत जातो.",
    "तो घरी गेला होता.",
    "माझे नाव रमेश आहे.",
    "तो घरि गेला होता.",   # intentional misspelling
]

for sentence in sentences:
    tokens = tokenizer.tokenize(sentence)
    print(f"Input : {sentence}")
    print(f"Tokens: {tokens}")
    print(f"Count : {len(tokens)}")
    print("─" * 50)

Input : मी शाळेत जातो.
Tokens: ['मी', 'शाळेत', 'जातो', '.']
Count : 4
──────────────────────────────────────────────────
Input : तो घरी गेला होता.
Tokens: ['तो', 'घरी', 'गेला', 'होता', '.']
Count : 5
──────────────────────────────────────────────────
Input : माझे नाव रमेश आहे.
Tokens: ['माझे', 'नाव', 'रमेश', 'आहे', '.']
Count : 5
──────────────────────────────────────────────────
Input : तो घरि गेला होता.
Tokens: ['तो', 'घर', '##ि', 'गेला', 'होता', '.']
Count : 6
──────────────────────────────────────────────────


In [8]:
sentence = "तो घरी गेला होता."
inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

last_hidden = outputs.last_hidden_state
print(f"Sentence        : {sentence}")
print(f"Embedding shape : {last_hidden.shape}")
print(f"  → batch size  : {last_hidden.shape[0]}")
print(f"  → token count : {last_hidden.shape[1]}")
print(f"  → hidden size : {last_hidden.shape[2]}")

Sentence        : तो घरी गेला होता.
Embedding shape : torch.Size([1, 7, 768])
  → batch size  : 1
  → token count : 7
  → hidden size : 768


In [10]:
import torch.nn.functional as F

correct    = "तो घरी गेला होता."
misspelled = "तो घरि गेला होता."

embeddings = []
for text in [correct, misspelled]:
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    cls = outputs.last_hidden_state[0][0]
    embeddings.append(cls)

sim = F.cosine_similarity(embeddings[0].unsqueeze(0), embeddings[1].unsqueeze(0))
print(f"Correct   : {correct}")
print(f"Misspelled: {misspelled}")
print(f"Cosine similarity: {sim.item():.4f}")
print()
print("→ >0.95 : MuRIL handles misspelling gracefully — Layer 2 can rank suggestions by context")
print("→ <0.90 : MuRIL sensitive to error — Layer 2 needs heavier correction work")

Correct   : तो घरी गेला होता.
Misspelled: तो घरि गेला होता.
Cosine similarity: 0.9999

→ >0.95 : MuRIL handles misspelling gracefully — Layer 2 can rank suggestions by context
→ <0.90 : MuRIL sensitive to error — Layer 2 needs heavier correction work


## MuRIL Findings — Day 3

### Key Results
- Cosine similarity correct vs misspelled: 0.9999
- MuRIL is robust to single-character Devanagari errors
- Misspelled words split into subwords: घरि → ['घर', '##ि']
- Known words stay intact: घरी → ['घरी']

### Architectural Decision Confirmed
- MuRIL role = contextual suggestion RANKER, not error detector
- Layer 1 handles detection (wordlist + rules)
- Layer 2 handles ranking (MuRIL embeddings)

### Layer 2 Plan
- Take Layer 1 candidates [घर, घरी]
- For each candidate, substitute into original sentence
- Score each substitution using MuRIL embeddings
- Return highest scoring candidate as top suggestion

### Next Step
- Implement candidate scoring function using MuRIL
- Integrate into /suggest endpoint as optional ML ranking